### `phi_functions.ipynb` 
*Created June 13, 2026*

This notebook implements both the scalar-valued and matrix-valued phi functions $\phi_0,\phi_1,\phi_2,\ldots$. 
Recall that for a nonnegative integer $m$ and an $N \times N$ matrix $B$ with complex entries, we define  
\begin{align*}
   \varphi_m(B) &= \sum_{k=0}^{\infty} \frac{B^k}{(k+m)!} \\[5pt]
                &= B^{-m}\left(e^B - I - B - \frac{B^2}{2!} - \cdots - \frac{B^{m-1}}{(m-1)!} \right)   \qquad \textrm{when } B \textrm{ is invertible}
\end{align*}

There are several to compute $\varphi_m(B)$: 
- **The explicit formula:** $\phi_m(B) = B^{-m}\left(e^B - I - B - \frac{B^2}{2!} - \cdots - \frac{B^{m-1}}{(m-1)!} \right)$

- **The recursive formula:** $\phi_{m+1}(B) = B^{-1}(\phi_m(B) - \frac{1}{m!}I)$
 
- **The Augmented Matrix Exponential:** Let $C_k$ be the $N(k+1) \times N(k+1)$ block diagonal matrix $C_k := \textrm{diag}(B,I^{(1)},...,I^{(k)})$, where the $I^{(j)}$'s are $N \times N$ identity blocks. Then the first $N$ rows of the matrix $e^C$ is exactly the block matrix $[\phi_0(B) \; \phi_1(B) \; \cdots \; \phi_k(B)]$.  

- **Kyrlov subspaces** (to compute the matrix-vector products $\phi_m(B) v$): [NEED TO LEARN THIS]

The explicit and recursive formulas have the downside of only working when $B$ is invertible. Also, inverting $B$ can be numerically unstable when $B$ is ill-conditioned. The augmented matrix exponential method is better than the explicit and recursive formulas because it works for invertible $B$, and it is numerically stable. 

The Krylov subspace approach is probably the best of the four because it's numerically stable, but cheapter than the augmented matrix exponential. 

In [1]:
using LinearAlgebra
using Random
using Statistics
# @time using NBInclude
# @nbinclude("..//set_makie_defaults.ipynb")

In [6]:
function phis(z::Number, k::Int64)
    """
    Given z (a real or complex number) and k (a nonnegative integer), compute ϕ_0(z), ϕ_1(z),...,ϕ_k(z) (all scalar quantities). 

    PARAMETERS
    ----------
    z :: a complex number 
    k :: a nonnegative integer

    RETURNS
    -------
    [ϕ_0(z),ϕ_1(z),…,ϕ_k(z)]    (vector of length k + 1) 
    """
    
    #Exception handling 
    k ≥ 0 || throw(ArgumentError("k must be nonnegative. Passed k = $k."))
    k ≤ 6 || throw(ArgumentError("To be safe, it is required that k ≤ 6."))
    
    #Initialize vector to store the ϕ_j's 
    φ = Vector{typeof(exp(z))}(undef, k+1)

    #Compute ϕ_0(z) = e^z using built-in exp() function 
    φ[1] = exp(z)  

    #If k = 0, we're done
    if k == 0
        return φ    
    end 

    ################################################################
    #If |z| is small, ϕ_j(z), j = 1,...,k using Taylor series approx
    ################################################################
    if abs(z) ≤ 1e-3    
        M = 5  #Use Taylor expansion with M + 1 terms 
        powers = [z^j for j=0:M]
        coeffs = 1 ./ [factorial(j) for j=0:k+M]
        
        for j = 1:k
            φ[j+1] = dot(coeffs[j+1:j+M+1], powers)    
        end 

        return φ    
    end 

    ####################################################
    #If |z| is not small, use recursive formula instead
    ####################################################   
    for m=1:k
        φ[m+1] = (φ[m] - 1/factorial(m-1)) / z    
        
    end 

    #### NOTE! ###
    #Because of the indexing, the above recursive formula looks wrong, but it's not! 
    #We have ϕ[m+1] = ϕ_m(z) and ϕ[m] = ϕ_{m-1}(z) for m = 1,...,k, and hence 
    #ϕ_m(z) = (ϕ_{m-1}(z) - 1/(m-1)!) / z , which is correct! 
    
    return φ
end 

phis (generic function with 3 methods)

In [7]:
function phis(B::Diagonal, k::Int64) 
    """
    Given a diagonal matrix A = diag(λ_1,...,λ_n), computes φ_m(A) = diag(φ_m(λ_1),...,φ_m(λ_n)) for m = 1,...,k. 

    PARAMETERS
    ----------
    B :: a diagonal matrix (must be of type Diagonal--see LinearAlgebra.jl docs for details on this type) 
    k :: a nonnegative integer; uses φ_0,...,φ_k. 

    RETURNS
    -------
    phi_mats :: A list of k + 1 matrices: φ_0(B),...φ_k(B) 
    """

    #Exception handling 
    k ≥ 0 || throw(ArgumentError("k must be nonnegative!"))
    k ≤ 4 || throw(ArgumentError("k must be less than or equal to 4 (for safety :))"))

    #Create vector of diagonal elements: λs = [λ_1,...,λ_N] 
    λs = B.diag

    #Apply `phis()` to each λ 
    phi_diags = phis.(λs, k)         #phi_diags = [ [φ_0(λ_1),...ϕ_k(λ_1)],..., [φ_0(λ_N),...ϕ_k(λ_N)]]

    #concatenate vecs side-by-side
    phi_diags_cat = hcat(phi_diags...)      #Create N x (k+1) matrix whose mth row is [ϕ_m(λ_1),...,ϕ_m(λ_N)]

    #Create the diagonal matrices Diag([ϕ_m(λ_1),...,ϕ_m(λ_N)])
    phi_mats = [Diagonal(phi_diags_cat[m,:]) for m = 1:k+1]
    
    return phi_mats

end     

phis (generic function with 3 methods)

In [8]:
function phis(B::AbstractMatrix, k::Int64)
    """
    Computes φ_0(B),...,φ_k(B) for an arbitrary square matrix B (with real or complex entries) and a nonnegative integer k. 
    
    PARAMETERS
    ----------
    B :: An N x N matrix of real or complex entries
    k :: A nonnegative integer
    
    RETURNS
    --------
    """
    #Error handling
    size(B,1) == size(B,2) || throw(ArgumentError("B must be a square matrix. Passed B of size $(size(B))."))
    0 ≤ k ≤ 4 || throw(ArgumentError("k must be a nonnegative integer between 0 and 4."))

    #Check if B is diagonal
    if B isa Diagonal
        return phis_diagonal(B, k)
    end 
    
    #Matrix type conversion (to be safe) 
    T = promote_type(eltype(B), Float64)
    B = Matrix{T}(B)      

    #Create big matrix 
    N = size(B,1) 
    C = zeros(T, N*(k+1), N*(k+1))
    C[1:N, 1:N] .= B 

    Iₙ = Matrix{T}(I, N, N)

    for j=1:k
        C[(j-1)*N+1:j*N, j*N+1:(j+1)*N] .= Iₙ
    end

    E = exp(C) 
    phi_mats = [E[1:N, j*N+1:(j+1)*N] for j=0:k]
    return phi_mats
end 

phis (generic function with 3 methods)

In [ ]:
# #Scalar phi's 
# phi0(z) = exp(z)
# phi1(z) = expm1(z) / z 
# phi2(z) = (expm1(z) - z) / z^2 
# phi3(z) = (expm1(z) - z - z^2/2) / z^3
# phi4(z) = (expm1(z) - z - z^2/2 - z^3/6) / z^4

# #Matrix phi's 
# phi0(B::AbstractMatrix) = exp(B)
# phi1(B::AbstractMatrix) = inv(B)*(exp(B) - I)
# phi2(B::AbstractMatrix) = (inv(B))^2 * (exp(B) - I - B)
# phi3(B::AbstractMatrix) = (inv(B))^3 * (exp(B) - I - B - B^2/2)
# phi4(B::AbstractMatrix) = (inv(B))^4 * (exp(B) - I - B - B^2/2 - B^3/6)

In [ ]:
# function compare_phi_funcs(;N, samples)

#     diff = 0.0 
    
#     for j=1:samples
#         B = rand(N, N)
#         #B = Diagonal(rand(N))
        
#         φ_0, φ_1, φ_2, φ_3, φ_4 = phi_matrices(B, 4)
#         diff += (1/4) * (norm(φ_1 - phi1(B))/norm(phi1(B)) + norm(φ_2 - phi2(B))/norm(phi2(B)) + 
#                          norm(φ_3 - phi3(B))/norm(phi3(B)) + norm(φ_4 - phi4(B))/norm(phi4(B)))    
#     end 

#     return diff / samples
# end 

In [ ]:
#@time compare_phi_funcs(N = 64, samples = 100)